# Imports

In [1]:
"""pip install tensorflow"""

'pip install tensorflow'

In [19]:
import numpy as np
import pandas as pd
from calc_rates import calc_rates
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input

print('TensorFlow:', tf.__version__)

TensorFlow: 2.21.0


Immportar todos los datos. Para tener medias bien hechas en el uso de campeones y winrate luego se desecharán los primeros archivos que además contienen aujeros importantes de datos.

In [3]:
"""
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
# Leer todos los archivos
dfs = []

#Los primeros años tienen algunas carencias de datos y 2021 es un año con muchos.
for year in range(2018,2022):
    df = pd.read_csv(f"../Data/Procesed data/Re-formulated data/{year}_LoL_esports_match_data_from_OraclesElixir.csv",low_memory=False)
    dfs.append(df)

# Unir train
train_set = pd.concat(dfs, ignore_index=True)

dfs = []
for year in range(2022,2024):
    df = pd.read_csv(f"../Data/Procesed data/Re-formulated data/{year}_LoL_esports_match_data_from_OraclesElixir.csv",low_memory=False)
    dfs.append(df)

# Unir val
val_set = pd.concat(dfs, ignore_index=True)

dfs = []
for year in range(2024,2027):
    df = pd.read_csv(f"../Data/Procesed data/Re-formulated data/{year}_LoL_esports_match_data_from_OraclesElixir.csv",low_memory=False)
    dfs.append(df)

# Unir test
test_set  = pd.concat(dfs, ignore_index=True)


train_set = calc_rates(train_set)
val_set = calc_rates(val_set)
test_set = calc_rates(test_set)

train_set.to_csv("../Data/Net/train_data.csv",index=False)
val_set.to_csv("../Data/Net/val_data.csv",index=False)
test_set.to_csv("../Data/Net/test_data.csv",index=False)
"""

'\nfrom sklearn.preprocessing import StandardScaler\nscaler = StandardScaler()\n# Leer todos los archivos\ndfs = []\n\n#Los primeros años tienen algunas carencias de datos y 2021 es un año con muchos.\nfor year in range(2018,2022):\n    df = pd.read_csv(f"../Data/Procesed data/Re-formulated data/{year}_LoL_esports_match_data_from_OraclesElixir.csv",low_memory=False)\n    dfs.append(df)\n\n# Unir train\ntrain_set = pd.concat(dfs, ignore_index=True)\n\ndfs = []\nfor year in range(2022,2024):\n    df = pd.read_csv(f"../Data/Procesed data/Re-formulated data/{year}_LoL_esports_match_data_from_OraclesElixir.csv",low_memory=False)\n    dfs.append(df)\n\n# Unir val\nval_set = pd.concat(dfs, ignore_index=True)\n\ndfs = []\nfor year in range(2024,2027):\n    df = pd.read_csv(f"../Data/Procesed data/Re-formulated data/{year}_LoL_esports_match_data_from_OraclesElixir.csv",low_memory=False)\n    dfs.append(df)\n\n# Unir test\ntest_set  = pd.concat(dfs, ignore_index=True)\n\n\ntrain_set = calc_rates

Tras haber ejecutado una vez el anterior, los datos se pueden cargar rápidamente con el siguiente codigo

In [4]:
train_set = pd.read_csv("../Data/Net/train_data.csv")
val_set = pd.read_csv("../Data/Net/val_data.csv")
test_set = pd.read_csv("../Data/Net/test_data.csv")

# Separación de features con labels

La label es la columna result. Además hay que deshacerse de unas cuantas columnas. Como los nombres de los jugadores y equipos que se han usado para clasificar las últimas métricas.

In [5]:
def remove_labels(df, label_name):
    X = df.drop(label_name, axis=1)
    y = df[label_name].copy()
    return (X, y)

In [6]:
train_set.info()

<class 'pandas.DataFrame'>
RangeIndex: 73820 entries, 0 to 73819
Data columns (total 98 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   gameid                      73820 non-null  str    
 1   side                        73820 non-null  bool   
 2   p1_name                     73604 non-null  str    
 3   p1_champion                 73820 non-null  str    
 4   p1_champion_id              73820 non-null  int64  
 5   p1_champion_last30_winrate  73820 non-null  float64
 6   p1_kills                    73820 non-null  int64  
 7   p1_deaths                   73820 non-null  int64  
 8   p1_pentakill                73820 non-null  float64
 9   p1_assist                   73820 non-null  int64  
 10  p1_dpm                      73820 non-null  float64
 11  p1_cspm                     73518 non-null  float64
 12  p1_vspm                     73820 non-null  float64
 13  p1_firstblood               73578 non-null

In [7]:
for i in range(1,6):
    train_set = train_set.drop(f"p{i}_name",axis=1)
    train_set = train_set.drop(f"p{i}_champion",axis=1)
    
train_set = train_set.drop("gameid",axis=1)
train_set = train_set.drop("team",axis=1)

for i in range(1,6):
    val_set = val_set.drop(f"p{i}_name",axis=1)
    val_set = val_set.drop(f"p{i}_champion",axis=1)

val_set = val_set.drop("gameid",axis=1)
val_set = val_set.drop("team",axis=1)

for i in range(1,6):
    test_set = test_set.drop(f"p{i}_name",axis=1)
    test_set = test_set.drop(f"p{i}_champion",axis=1)

test_set = test_set.drop("gameid",axis=1)
test_set = test_set.drop("team",axis=1)

champion_columns = [f"p{i}_champion_id"for i in range(1,6)]
train_set[champion_columns] = train_set[champion_columns].astype("category")
val_set[champion_columns] = train_set[champion_columns].astype("category")
test_set[champion_columns] = train_set[champion_columns].astype("category")

In [8]:
train_set.info()

<class 'pandas.DataFrame'>
RangeIndex: 73820 entries, 0 to 73819
Data columns (total 86 columns):
 #   Column                      Non-Null Count  Dtype   
---  ------                      --------------  -----   
 0   side                        73820 non-null  bool    
 1   p1_champion_id              73820 non-null  category
 2   p1_champion_last30_winrate  73820 non-null  float64 
 3   p1_kills                    73820 non-null  int64   
 4   p1_deaths                   73820 non-null  int64   
 5   p1_pentakill                73820 non-null  float64 
 6   p1_assist                   73820 non-null  int64   
 7   p1_dpm                      73820 non-null  float64 
 8   p1_cspm                     73518 non-null  float64 
 9   p1_vspm                     73820 non-null  float64 
 10  p1_firstblood               73578 non-null  float64 
 11  p1_firstbloodassist         73820 non-null  float64 
 12  p1_goldat10                 73820 non-null  float64 
 13  p1_goldat15                

In [24]:
X_train, y_train_prep = remove_labels(train_set, 'result')
X_val, y_val_prep = remove_labels(val_set, 'result')
X_test, y_test_prep = remove_labels(test_set, 'result')

In [10]:
assert(len(X_train) == len(y_train))
print("Entrenamiento:",len(X_train))

assert(len(X_val) == len(y_val))
print("Val:",len(X_val))

assert(len(X_test) == len(y_test))
print("Prueba:",len(X_test))

Entrenamiento: 73820
Val: 47270
Prueba: 52980


In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

chp_train = X_train[[f"p{i}_champion_id" for i in range(1,6)]].copy()
chp_val = X_val[[f"p{i}_champion_id" for i in range(1,6)]].copy()
chp_test = X_test[[f"p{i}_champion_id" for i in range(1,6)]].copy()

X_train = X_train.drop((f"p{i}_champion_id" for i in range(1,6)),axis=1)
X_val = X_val.drop((f"p{i}_champion_id" for i in range(1,6)),axis=1)
X_test = X_test.drop((f"p{i}_champion_id" for i in range(1,6)),axis=1)

X_train_prep = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index)

X_val_prep = pd.DataFrame(
    scaler.fit_transform(X_val),
    columns=X_val.columns,
    index=X_val.index)

X_test_prep = pd.DataFrame(
    scaler.fit_transform(X_test),
    columns=X_test.columns,
    index=X_test.index)

for i in range(1,6):
    X_train_prep.insert(1+(16*(i-1)),(f"p{i}_champion_id"),chp_train[f"p{i}_champion_id"])
    X_val_prep.insert(1+(16*(i-1)),(f"p{i}_champion_id"),chp_val[f"p{i}_champion_id"])
    X_test_prep.insert(1+(16*(i-1)),(f"p{i}_champion_id"),chp_test[f"p{i}_champion_id"])

In [12]:
X_train_prep.head()

,side,p1_champion_id,p1_champion_last30_winrate,p1_kills,p1_deaths,p1_pentakill,p1_assist,p1_dpm,p1_cspm,p1_vspm,...,p5_xpat15,dragons,barons,heralds,towers,goldat10,goldat15,xpat10,xpat15,wins_vs_opponent
0,1.0,119,-1.100477,0.131863,0.074523,-0.028282,2.171872,-0.047952,-0.658826,0.508362,...,0.302072,-0.791633,-0.918071,-0.911652,-1.045967,0.051586,-0.011320,0.343358,0.386570,-0.750849
1,-1.0,21,-1.100477,1.416102,1.100875,-0.028282,0.800695,0.855966,0.904908,0.204514,...,0.583836,0.600220,3.206321,-0.911652,1.355176,0.244388,0.352293,0.478172,0.675136,-0.750849
2,1.0,138,-1.100477,-0.724297,-0.438654,-0.028282,1.623401,-0.303501,1.610997,0.204514,...,0.583836,-0.791633,-0.918071,-0.911652,1.088382,0.244388,0.352293,0.478172,0.675136,-0.479289
3,-1.0,119,-1.100477,-0.724297,0.074523,-0.028282,0.252224,-0.082128,0.497947,0.508362,...,0.302072,0.600220,1.831524,-0.911652,0.554795,0.051586,-0.011320,0.343358,0.386570,-0.750849
4,1.0,37,-1.100477,-0.724297,-0.951830,-0.028282,-0.570482,0.232843,0.857442,0.220387,...,0.789759,-0.791633,-0.918071,-0.911652,-0.779173,-0.203742,-0.197612,0.300297,0.431013,-0.750849


In [13]:
X_train_prep.describe()

,side,p1_champion_last30_winrate,p1_kills,p1_deaths,p1_pentakill,p1_assist,p1_dpm,p1_cspm,p1_vspm,p1_firstblood,...,p5_xpat15,dragons,barons,heralds,towers,goldat10,goldat15,xpat10,xpat15,wins_vs_opponent
count,73820.000000,7.382000e+04,7.382000e+04,7.382000e+04,7.382000e+04,7.382000e+04,7.382000e+04,7.351800e+04,7.382000e+04,7.357800e+04,...,7.382000e+04,7.382000e+04,7.382000e+04,7.382000e+04,7.382000e+04,7.382000e+04,7.382000e+04,7.382000e+04,7.382000e+04,7.382000e+04
mean,0.000000,2.340883e-16,-5.544197e-17,-8.624306e-17,-1.463052e-17,8.624306e-17,-2.279281e-16,1.583494e-15,2.032872e-16,-6.180480e-18,...,2.710496e-16,1.170442e-16,-1.078038e-16,-2.464088e-17,7.161254e-17,-1.767983e-15,6.468230e-16,2.229999e-15,-1.712541e-15,4.928175e-17
std,1.000007,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,...,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00,1.000007e+00
min,-1.000000,-1.100477e+00,-1.152377e+00,-1.465007e+00,-2.828216e-02,-1.393188e+00,-2.592464e+00,-6.141890e+00,-3.779563e+00,-3.059821e-01,...,-3.534099e+00,-1.487559e+00,-9.180712e-01,-9.116517e-01,-1.579554e+00,-4.605952e+00,-4.185672e+00,-7.096957e+00,-6.413461e+00,-7.508491e-01
25%,-1.000000,-1.100477e+00,-7.242968e-01,-9.518305e-01,-2.828216e-02,-8.447169e-01,-7.162052e-01,-6.717277e-01,-6.482541e-01,-3.059821e-01,...,-6.758100e-01,-7.916329e-01,-9.180712e-01,-9.116517e-01,-1.045967e+00,-6.701446e-01,-6.878863e-01,-6.038879e-01,-6.401639e-01,-7.508491e-01
50%,0.000000,1.041404e-01,-2.962170e-01,7.452251e-02,-2.828216e-02,-2.201090e-02,-1.428937e-01,-3.507212e-03,-3.986121e-02,-3.059821e-01,...,-2.835216e-02,-9.570640e-02,4.567263e-01,4.515416e-01,2.880013e-01,-1.273469e-01,-1.207367e-01,1.019298e-01,5.914816e-02,-4.792885e-01
75%,1.000000,7.091622e-01,5.599425e-01,5.876990e-01,-2.828216e-02,5.264598e-01,5.805961e-01,6.631802e-01,5.503534e-01,-3.059821e-01,...,5.855484e-01,6.002201e-01,4.567263e-01,4.515416e-01,8.215887e-01,5.359561e-01,5.647507e-01,6.472958e-01,6.419082e-01,3.353931e-01
max,1.000000,1.612625e+00,8.693458e+00,7.258993e+00,3.535798e+01,6.285402e+00,6.409400e+00,4.387890e+00,8.573172e+00,3.268165e+00,...,8.380703e+00,4.079853e+00,7.330713e+00,1.814735e+00,1.355176e+00,9.067731e+00,8.357058e+00,7.878407e+00,5.283459e+00,7.395967e+00


# Definición del modelo

In [52]:
print(X_train_prep.shape)
print(y_train_prep.unique())

(73820, 85)
[0 1]
